Nei sistemi distribuiti le componenti comunicano tramite **ricezione e invio di messaggi**.  
Realizzare lo scambio non è banale, servono dettagli come rappresentazione dei dati, delimitazione dei messaggi, trasmissione, rilevamento errori etc..

Abbiamo già visto nel corso di reti come la soluzione sia l'**organizzazione a livelli** delle funzionalità, di modo da poter usare servizi ad alto livello senza dover gestire direttamente i dettagli sottostanti.  
Nel corso ci si concentrerà più che altro sul **middleware di comunicazione**, che offre alle applicazioni distribuite diverse funzionalità come:
- Chiamate a procedura remota **RPC**
- code di messaggi
- comunicazioni verso più destinatari (multicast a livello applicativo)

Esistono anche altri middleware oltre a quello di comunicazione come naming (per identificare e permettere la condivisione di risorse), security (per garantire comunicazioni sicure tra nodi del DS), di consenso, di locking (per evitare conflitti di accesso a risorse condivise) etc..  

Le funzionalità fornite dal middleware sono riutilizzabili e indipendenti dalla logica specifica dell'applicazione (es. un'applicazione può richiedere di eseguire un'operazione remota senza dover reimplementare da zero tutti i passaggi per poter trasmettere la richiesta).  
Ci focalizzeremo anzitutto su RPC, per poi passare a sistemi orientati ai messaggi e poi altri meccanismi di comunicazione ancora.

<img src="imgs/middleware.png" alt="middleware" width="400"/>

La comunicazione può essere classificata secondo tre criteri:
1. **Persistenza** == se il messaggio viene conservato o meno per una consegna successiva al destinatario (persistente vs transiente)
2. **Sincronizzazione** == se il mittente deve attendere, ed eventualmente fino a quale evento specifico deve aspettare
3. **Dipendenza temporale** == se i messaggi costituiscono unità discrete o una sequenza temporale (comunicazione discreta vs streaming)

Chiaramente in generale avere un criterio non implica necessariamente averne anche un altro.

**Comunicazione Persistente e Transiente**

Nella **persistente** *il middleware conserva il msg affinché possa esser consegnato successivamente* secondo le garanzie del servizio --> **il destinatario non deve esser necessariamente attivo quanto il mittente invia**.

es. $A$ invia un messaggio, il middleware lo prende in carico conservandolo, $B$ diventa attivo dopo un certo tempo e riceve il messaggio dal middleware.

esempi reali di comunicazione persistente sono email, chat e middleware orientati ai messaggi. (NB persistente non significa conservato per sempre, ma che il servizio permette la conservazione che serve a separare tempo di invio e ricezione)

Nella **transiente** al contrario non è offerta qeusta conservazione se il destinatario è offline --> il messaggio può essere scartato se le parti non sono entrambe attive.  
Possono comunque esistere *buffer temporanei*: la differenza è che non c'è un servizio middleware che garantisca la consegna al destinatario quando sarà attivo alla ricezione (es. nei router si memorizzano temporaneamente i pacchetti, ma ciò non garantisce che siano tenuti fino a disponibilità del destinatario).

**Sincrona e Asincrona**

Nella comunicazione **sincrona** *l'operazione resta bloccata fino a un determinato evento di completamento*.  
Ciò ovviamente non significa che tutta la macchina si fermi, ma che almeno il thread che esegue l'operazione deve attendere. 

Esistono diversi punti di sincronizzazione:
- **At request submission** se il mittente deve attendere che sia il middleware a prendere in carico la richiesta
- **At request delivery** se si attende che la richiesta raggiunga il punto previsto dal destinatario
- **After request processing** se deve attendere pure che il destinatario elabori la richiesta e produca una risposta

Ciò implica che **un invio sincrono può terminare prima che l'applicazione destinataria abbia effettivamente eseguito l'operazione richiesta**. 

Un ACK, allo stesso modo, dobbiamo sempre chiederci *cosa sta confermando* (se ad es. la presa in carico dal middleware oppure l'effettivo arrivo del messaggio al destinatario).

<img src="imgs/sync.png" alt="sincrona" width="400"/>

Nel caso **asincrono** si parla di *invio non bloccante* in quanto il mittente lascia il msg al sistema di comunicazione e può continuare subito il proprio lavoro.  
Il middleware può mantenere temporaneamente il msg mentre ne gestisce la trasmissione per favorire eventualmente concorrenza con altri nodi (nota: non necessariamente fino a che il destinatario è attivo, non necessariamente è comunicazione persistente!). 

La **ricezione** a sua volta può essere *bloccante* se il ricevente deve attendere che arrivi un certo msg, oppure *non bloccante* se egli può controllare se gli è arrivato o meno (polling).  
**Il fatto che il mittente invii in modo asincrono non obbliga il destinatario a riceve in modo non bloccante**, dipende dalla logica dell'applicazione.

Il vantaggio di asincronia da entrambi i lati è che disaccoppia (**decouples**) mittente e destinatario, permettendo quindi concorrenza e responsiveness maggiori.

**Discreta e Streaming**

Con la comunicazione **discreta** ogni messaggio rappresenta un'unità informativa distinta (es. richiesta di interrogazione di un database o msg di notifica). Può esserci un flusso logico tra messaggi, ma non formano necessariamente un flusso da riprodurre secondo un certo ritmo temporale.

Con la **streaming** l'informazione è trasmessa tramite *sequenza di messaggi con relazioni temporali rilevanti* (es. audio e video: per riprodurre correttamente il contenuto serve ricostruire la sequenza rispettando il ritmo temporale) --> **non basta ricevere dati isolati, ma bisogna sapere come ricomporli nel tempo**.

#### Combinare persistenza e sincronizzazione
Queste due proprietà sono indipendenti e producono 4 combinazioni principali

**Persistente e asincrona** --> il mittente invia il messaggio e continua a lavorare. Il middleware conserva il messaggio finché il destinatario può riceverlo. Questo è proprio il modello già anticipato tipo email, chat o sistemi orientati a messaggistica persistente.

**Persistente e sincrona** --> il middleware conserva il messaggio, ma l'utente *attende una conferma prima di proseguire*.  
Nella figura il msg è conservato in particolare lato destinatario quando $B$ è ancora inattivo. Non appena $A$ riceve l'ACK dal middleware, può proseguire (in questo caso quindi sincronia **at request delivery** in quanto l'ack riguarda il fatto che il msg è stato salvato lato destinatario, non elaborazione o semplice presa a carico dal middleware in locale)

<img src="imgs/persistente-asincrona.png" alt="persistente-asincrona" width="500"/>

**Transiente e asincrona** --> il mittente invia senza attendere, ma manca la conservazione del messaggio per futura disponibilità del destinatario.  
Se il destinatario è inattivo allora niente da fare, il msg può andare perso (es. UDP). Il mittente prosegue il suo lavoro senza sapere che il messaggio sia arrivato. 

<img src="imgs/ta.png" alt="transiente-asincrona" width="300"/>

**Transiente e sincrona** --> il mittente attende un certo evento ma il sistema non conserva il msg per una consegna in differita. Tre varianti in questo caso, in base a *quando il mittente riprende a lavorare*:
- **Receipt-based** se riprende a lavorare appena il messaggio raggiunge il buffer del destinatario
- **Delivery-based** se appena il messaggio viene passato all'applicazione destinataria
- **Response-based** se appena il destinatario ha elaborato il messaggio e prodotto una risposta (es. RPC)

La terza variante è quella classica a cui tipicamente associamo tipica richiesta-risposta sincrona.

<img src="imgs/ts.png" alt="transiente-sincrona" width="500"/>

<img src="imgs/ts2.png" alt="transiente-sincrona" width="300"/>

## Communication Failures
In generale quando un client esegue una chiamata RPC avviene quanto segue: il client manda richiesta, il server esegue la procedura specifica per poi inviare il risultato, e il client lo riceve. 

<img src="imgs/RPC.png" alt="rpc" width="200"/>

Approfondiremo per bene come funziona ogni singolo passaggio in questione, ma intanto è importante capire che il middleware *nasconde parte dei dettagli di comunicazione*, ma **non elimina i problemi dovuti al fatto che ci troviamo in un ambiente distribuito**

Esistono diversi fallimenti possibili in termini di comunicazione:
- **Message loss or delay** (es. la richiesta o la risposta sono state perse o ritardate o la connessione è stata interrotta..)
- **Server crash**: può avvenire 
  - prima di processare la richiesta
  - dopo/durante il processamento

  <img src="imgs/server-crash.png" alt="server crash" width="200"/>  

Chiaramente il client non è in grado di sapere in che caso si trovi il server e in generale quindi ovviamente il fatto che una risposta non arrivi, non implica che l'operazione non sia stata eseguita (importante ad esempio se chiediamo di incrementare un contatore... rieseguire l'operazione sarebbe problematico)
- **Client crash** dopo aver inviato la richiesta

(*vedremo come questi problemi si presentano non solo nel caso RPC, ma anche con le code di messaggi e i sistemi pub/sub*)

In generale sistemi diversi possono gestire queste problematiche in modo diverso, più precisamente **offrendo semantiche diverse**.  
Una **semantica in presenza di guasti** indica infatti **quali garanzie offre il sistema nonostante questi problemi**. 

Le semantiche si possono applicare sia sulla consegna dei messaggi (es. MOM) che sull'elaborazione della richiesta, in questo caso ci concentriamo sulle **service processing** con *sender=client e receiver=server* (e quindi garanzie sul fatto che l'elaborazione sia avvenuta una o più volte..).

Quattro semantiche, dalle meno "garanti" alle migliori (ma anche più costose e complesse da implementare!).
1. **May-be**: non si garantisce che la richiesta venga eseguita
2. **At-least-once**: si garantisce l'esecuzione della richiesta, che però può essere avvenuta più volte. Attenzione al fatto che questo è vero se il client riceve la risposta, sennò in generale **non è detto che l'operazione sia stata eseguita** perché magari il server è down --> se ricevo risposta sicuro $\ge 1$ esecuzioni, se non ricevo risposta potrebbe non essere mai stata eseguita, oppure potrebbe essere stata eseguita $\ge 1$ volte.
3. **At-most-once**: si garantisce che al massimo è stata eseguita una singola elaborazione della richiesta. Se il client riceve una risposta di successo --> è stata eseguita esattamente una volta, se non riceve risposta conclusiva non può saperlo con certezza ma saprà sempre che più di una volta non è stata eseguita. 
4. **Exactly-once**: anche qui come in at-most-once se il client non riceve risposta ha come unica certezza che al più l'operazione è stata eseguita una sola volta.  
Ciò che rende questa semantica migliore di at-most-once è che *mentre in at-most-once se arriva una richiesta al server non c'è garanzia che venga completata* (ma comunque si garantisce che non sarà processata più di una volta), **in exactly-once esistono meccanismi aggiuntivi e più complessi che permettono, una volta arrivata la richiesta, di gestire complicazioni come guasti, retries o ritardi completando sempre e comunque l'operazione** (o almeno, sotto le condizioni garantite dal protocollo).

In pratica cioè le semantiche ragionano dal momento in cui la richiesta arriva (a patto che il server prima o poi torni attivo ovviamente): at least once ci dice che se gli è arrivata la processa almeno una volta, at most once che se gli è arrivata la processa al massimo una volta, exactly once che se gli è arrivata la processa esattamente una volta nei limiti del protocollo.

**I tre strumenti principali**

I tre meccanismi di base per implementare le prime semantiche sono:
- **RR1 - Request Retry**: il client quando invia la richiesta avvia anche un timer. Se non riceve risposta entro un timeout --> ritrasmette.  
Chiaro che il timeout segna attesa eccessiva ma non è detto che il server fosse guasto (magari richiesta lunga da elaborare o server overloaded di per sé). Si possono limitare i tentativi o aumentare progressivamente il timeout per evitare di sovraccaricare il server (idea del *backoff*, eventualmente esponenziale);
- **DF - Duplicate Filtering**: il server riconosce che una richiesta è ritrasmissione di una precedente e quindi **non la esegue nuovamente**
- **RR2 - Result Retransmit**: il server conserva il risultato già calcolato e, se arriva un duplicato della richiesta, *ritrasmette il risultato in questione senza dover ripetere l'operazione*.  
Ciò risolve un problema per cui il solo Duplicate Filtering non basta: ignorare tutti i duplicati evita la doppia esecuzione, ma se il client non ha mai ricevuto la risposta --> se non gliela mandiamo resta senza niente porello

### May-be
Con questa semantica non servono gli strumenti sopra descritti: è il minimo livello di garanzie. Il client che manda la richiesta, se non riceve risposta non ha alcuna garanzia su ciò che è successo (es. richiesta persa e non si esegue nulla, il server esegue ma si perde la risposta, il server crasha durante l'elaborazione...)

Il **vantaggio** è il basso costo dei meccanismi di comunicazione, lo **svantaggio** è che non ci sono garanzie di alcun tipo per operazioni importanti.  

Tra le possibili applicazioni: dati di sensori per i quali perdite sporadiche sono tollerabili; logging non critico.

<img src="imgs/may-be.png" alt="may be" width="500"/>

### At-least-once
Si presenta un modello dove il client usa **RR1** mentre il server non usa né DF né RR2.  
Se la risposta non arriva il client riporva, e il server tratta ogni richiesta ricevuta come un'operazione da eseguire. Nell'esempio **l'operazione viene eseguita due volte**, ma il client non può capirlo dalla risposta ricevuta.

Ci troviamo in at-least-once proprio perché il client se riceve risposta di successo sa che l'operazione è stata eseguita almeno una volta, mentre se non riceve risposta non può sapere se sia mai stata eseguita (magari il server è morto) o se è stata eseguita più di una volta!

<img src="imgs/at-least-once.png" alt="at least once" width="300"/>

Questa modalità è tipicamente adottata nei **servizi idempotenti**, ovvero quei servizi per cui un'operazione, se eseguita più volte con lo stesso input, danno ancora lo stesso risultato (es. get o put di una chiave in un database).  
Al contrario un es. di operazione non idempotente è l'incremento di un contatore.

### At-most-once
Qui la richiesta viene eseguita zero oppure una volta.  
Nel modello che vediamo si combinano **RR1** lato client e **DF**, **RR2** lato server.

Nell'esempio la prima richiesta si perde --> client ritrasmette, quindi il server registra la richiesta, la esegue e conserva il risultato. La risposta però si perde --> il client trasmette di nuovo. Il server riconosce il duplicato e quindi **non riesegue l'operazione** ma restituisce il risultato già calcolato.

<img src="imgs/at-most-once.png" alt="at most once" width="300"/>

Questo permette di gestire anche operazioni non idempotenti (es. incrementi o addebiti), ma se il client non riceve risposta ricordiamo che non può sapere se l'operazione è stata eseguita o meno (sa solo con certezza che al massimo è stata eseguita una volta).

Chiaramente per **DF** e **RR2** è necessario che *il server sappia riconoscere richieste duplicate e associarle a un risultato*.  
Per fare ciò il client include un ID univoco (sia questo **xid**) per ogni nuova richiesta, usando sempre quello per stesse richieste ritrasmesse in caso di timeout.  
Per quel che riguarda il server invece, questo quando riceve un certo xid per la prima volta esegue l'operazione e conserva il risultato associandolo a quello xid, in modo che se arrivi una nuova richiesta uguale possa rimandare direttamente il risultato.

<img src="imgs/at-most-once2.png" alt="at most once" width="200"/>

In generale per creare lo xid univoco per ogni nuova richiesta si fa qualcosa del tipo:
$$xid = (client\_id, nonce)$$
con nonce incrementale, mentre il server associa ad ogni xid una coppia:
$$xid \rightarrow (status, reply)$$
dove lo status è **pending** se il servizio sta ancora in esecuzione, **completed** se invece il processamento è terminato e il risultato (reply) è pronto in memoria per essere trasmesso.  
Se il duplicato arriva con pending (magari perché timeout richiesta piccolo rispetto al tempo di elaborazione del server, o server in sovraccarico) --> il server lo ignora o lo mette in coda in attesa che l'elaborazione termini.  
Se invece arriva con completed --> il server ritrasmette il risultato già calcolato.

Il server chiaramente non può salvare indefinitivamente ogni risposta. Per gestire la cosa si possono inserire timeout oppure finestre scorrevoli sugli xid (nella finestra quelli gestibili, fuori dalla finestra si scartano).

Il pattern appena visto nello specifico è chiamato **Idempotent Receiver**, comunque questi concetti torneranno nello studio di Kafka.



### Exactly-once
Qui i tre meccanismi non bastano più: il problema in particolare emerge con i crash.

Supponiamo ad esempio che il server *esegua un addebito, ma crashi prima di registrare stabilmente che la richiesta è stata completata* (es. conservava le informazioni solo in memoria volatile). 

Allora dopo il riavvio potrebbe non sapere che l'addebito è stato avvenuto.  
**Se il client ritenta allora rischia di eseguirlo nuovamente --> ritrasmissione, filtraggio e cache del risultato non bastano più se il loro stato non sopravvive correttamente ai guasti e non è coordinato con gli effetti dell'operazione.**

Exactly-once in questo senso segue proprio la logica transazionale del "**tutto o niente**: o l'operazione viene completata una sola volta, oppure non deve produrre alcun effetto parziale. 

Ribadiamo la differenza rispetto a at-most-once: anche in at-most-once si garantisce che comunque anche se non ricevo risposta, se crasha tutto, se ti esplode casa, a patto che il server torni attivo allora la richiesta sarà stata eseguita al massimo una volta. Tuttavia at-most-once accetta anche di perdere un'operazione nel caso ci sia un problema durante l'esecuzione, pur di non ripeterla.  
**Exactly-once al contrario promette anche che, se il guasto avviene quando la richiesta è già stata ricevuta/è in corso d'esecuzione nel server --> il completamento sarà garantito grazie a meccanismi aggiuntivi.**

In generale in realtà è veramente difficile ottenere la forma più forte di questa garanzia 

#### Gli strumenti aggiuntivi
Per ottenere questa semantica servono dei meccanismi più ulteriori: 
- **Replicazione del server**: più server mantengono copie del servizio e del suo stato --> un guasto può esser gestito facendo proseguire un'altra replica (es. schemi come leader-followers o primary-backup).  
La difficoltà qui è che siamo in un sistema distribuito: *ad ogni operazione tutti i server devono aggiornare il loro stato in modo coerente* e *il fatto che ci siano duplicati deve essere trasparente al client* --> esiste una sezione dello studio di algoritmi distribuiti dedicata solo a questo
- **Meccanismi di Recovery**: se avviene un crash, il server deve essere in grado di recuperare lo stato corretto e coerente del servizio. Tra questi meccanismi:
  - **Checkpoint dello stato**: si salva in una memoria **persistente** lo stato del server ogni tot tempo
  - **Checkpoint con snapshot**: non ci si limita a un singolo server, ma si vuole salvare una visione globale e consistente anche dei replicati. Più complesso e richiede coordinamento tra server
  - **Write-Ahead Logging**: prima di eseguire un operazione, il server salva la richiesta in un log persistente

##### WAL (AKA **Commit Log**)
Il Write-Ahead Log (WAL), o Commit Log, è **un registro persistente delle operazioni necessarie a ricostruire lo stato del servizio**.  

In generale il principio è il seguente: **prima si registra l'informazione necessaria al recupero, solo poi si applica l'operazione in sé**. 

Ogni cambio di stato è salvato come nuova entry del log, che è scritto in memoria persistente (es. disco o SSD).  
Il log è scritto in modalità **append-only**, ossia le nuove operazioni sono aggiunte alla sua coda senza mai modificare quelle già presenti e conservando l'ordine.

Es. il client chiede `put(K4, V4)`. Allora il server aggiunge al log l'operazione che imposta $K4$ a $V4$ (`set(K4, V4)`) e SOLO ALLORA aggiorna la struttura chiave valore.  
*Lo scrivere prima nel log è ovviamente fondamentale di modo che se crashasse senza aver scritto il log ma avendo eseguito l'operazione (magari in RAM) allora perderebbe la modifica e non potrebbe recuperarla*.

<img src="imgs/wal.png" alt="wal" width="400"/>

WAL e Checkpoint differiscono in quanto *checkpoint conserva l'intero stato del sistema in un dato istante*, mentre *WAL è la specifica sequenza delle modifiche* (lo stato è da vedersi come lo stato risultante del server dopo le varie modifiche, che viene salvato ogni tot tempo. Il WAL invece registra singolarmente le operazioni che modificano lo stato). 

**Per recuperare da un crash quindi prima il server carica un checkpoint valido, per poi riprodurre le entry successive allo stato presenti nel WAL**. In tal caso nessuno stato è perso, *a patto che il log sia stato scritto in memoria persistente prima del crash*.

Infatti **scrivere direttamente su storage persistente è molto costoso, specie se viene fatto una entry alla volta** --> un'alternativa migliore è proprio accumulare entry in un buffer e scriverle in gruppi. Se però il server crasha prima che il buffer sia scritto su storage persistente so cazzi.

Per concludere, tra le quattro semantiche, **at-least-once e at-most-once sono le più diffuse**. At-least-once può essere preferita per la semplicità e scalabilità, specie se il servizio è *idempotente* e se l'applicazione tollera duplicati. 

In generale la scelta dipende dalle domande:
- Quali errori può tollerare l’applicazione: perdite, duplicati, attese?
- Quale costo si è disposti a sostenere per evitarli?